# BedCoverage WES ClinCNV Transparency Notebook

This notebook documents how we created:

- the cleaned BAF folder at `/mnt/myvolume/panel_seq/new_bed_analysis/baf_from_pair_vcfs_clean`
- the new WES BedCoverage-derived coverage matrices
- the ClinCNV runs in:
  - `/mnt/myvolume/panel_seq/new_bed_analysis/new_clincnv_runs/full_wes_s200_l9_f2_baf_bedcoverage_wes`
  - `/mnt/myvolume/panel_seq/new_bed_analysis/new_clincnv_runs/full_wes_s200_l9_f2_no_baf_bedcoverage_wes`

The goal is full transparency: the notebook points directly to the local scripts, shows their contents, records the file paths used, and summarizes the generated outputs.

In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path('/mnt/myvolume/panel_seq/new_bed_analysis')
RUNS_ROOT = BASE / 'new_clincnv_runs'

BAF_SCRIPT = BASE / 'baf_from_pair_vcfs.sh'
BAF_FOLDER = BASE / 'baf_from_pair_vcfs_clean'

OFFTARGET_SCRIPT = BASE / 'gen_wes_offtarget_bed_100kb.sh'
BEDCOVERAGE_SCRIPT = BASE / 'build_wes_bedcoverage_matrices.sh'

TARGET_BED = BASE / 'ssSC_v5.gc.genes.bed'
OFFTARGET_BED = BASE / 'clincnv_offtarget_wes_100kb_filtered.bed'

TUMOR_ONTARGET = BASE / 'tumor.ontarget.wes_mapq0_bedcoverage.cov'
NORMAL_ONTARGET = BASE / 'normal.ontarget.wes_mapq0_bedcoverage.cov'
TUMOR_OFFTARGET = BASE / 'tumor.offtarget.wes_100kb_mapq10_bedcoverage.cov'
NORMAL_OFFTARGET = BASE / 'normal.offtarget.wes_100kb_mapq10_bedcoverage.cov'

BEDCOVERAGE_CACHE = BASE / 'bedcoverage_wes_cache'
BEDCOVERAGE_LOGS = BASE / 'bedcoverage_wes_logs'

BafRun = RUNS_ROOT / 'full_wes_s200_l9_f2_baf_bedcoverage_wes'
NoBafRun = RUNS_ROOT / 'full_wes_s200_l9_f2_no_baf_bedcoverage_wes'

BAF_RUN_SCRIPT = BafRun / 'run_clincnv.sh'
NO_BAF_RUN_SCRIPT = NoBafRun / 'run_clincnv.sh'

def show_text(path, max_chars=None):
    text = Path(path).read_text()
    if max_chars is not None:
        text = text[:max_chars]
    print(text)

def show_head(path, n=5):
    with open(path, 'r', encoding='utf-8') as handle:
        for idx, line in zip(range(n), handle):
            print(line.rstrip('\n'))

def summarize_file(path):
    path = Path(path)
    return {
        'path': str(path),
        'exists': path.exists(),
        'size_bytes': path.stat().st_size if path.exists() else None,
    }


## 1. Cleaned BAF folder

The cleaned BAF folder was created from the pair VCFs with the script below. It reads `pairs_df_filtered.csv`, opens each `unfiltered.<tumor>__<normal>.vcf.gz`, keeps biallelic SNPs, requires valid `GT`, `AD`, and `DP`, enforces `DP >= 10`, and writes per-sample BAF tables into `baf_from_pair_vcfs_clean`.

Important implementation details from the script:

- source VCF root: `/mnt/myvolume/panel_seq/new_bed_analysis/vcfs`
- output BAF folder: `/mnt/myvolume/panel_seq/new_bed_analysis/baf_from_pair_vcfs_clean`
- per-sample duplicate loci across multiple pairs are collapsed by retaining the highest-depth row
- skipped or missing inputs are recorded in `skipped_pairs.tsv`

In [ ]:
show_text(BAF_SCRIPT)

In [ ]:
print('BAF folder exists:', BAF_FOLDER.exists())
tsv_files = sorted(BAF_FOLDER.glob('*.tsv'))
print('Number of BAF TSVs:', len(tsv_files))
print('First 10 TSV files:')
for path in tsv_files[:10]:
    print('  ', path.name)

print('\nREADME:')
show_text(BAF_FOLDER / 'README.md')

print('\nSkipped pairs log:')
show_head(BAF_FOLDER / 'skipped_pairs.tsv', n=20)

## 2. WES off-target BED and BedCoverage-derived matrices

The WES off-target BED was rebuilt into 100 kb bins, excluding target intervals and dropping bins smaller than 50 kb. That produced:

- `/mnt/myvolume/panel_seq/new_bed_analysis/clincnv_offtarget_wes_100kb_filtered.bed`

The coverage matrices were then built with `BedCoverage` using:

- on-target MAPQ `0`
- off-target MAPQ `10`
- target BED `ssSC_v5.gc.genes.bed`
- off-target BED `clincnv_offtarget_wes_100kb_filtered.bed`
- cached per-sample BED outputs under `bedcoverage_wes_cache`
- combined matrices written to the four `*.wes_*_bedcoverage.cov` files

The two scripts below contain the exact code used.

In [ ]:
show_text(OFFTARGET_SCRIPT)

In [ ]:
show_text(BEDCOVERAGE_SCRIPT)

In [ ]:
coverage_summary = pd.DataFrame([
    summarize_file(TARGET_BED),
    summarize_file(OFFTARGET_BED),
    summarize_file(TUMOR_ONTARGET),
    summarize_file(NORMAL_ONTARGET),
    summarize_file(TUMOR_OFFTARGET),
    summarize_file(NORMAL_OFFTARGET),
])
coverage_summary

In [ ]:
print('Target coverage header:')
show_head(TUMOR_ONTARGET, n=3)

print('\nOff-target coverage header:')
show_head(TUMOR_OFFTARGET, n=3)

print('\nBedCoverage cache directories:')
for path in sorted(BEDCOVERAGE_CACHE.iterdir()):
    print(path)

print('\nAvailable BedCoverage logs:')
for path in sorted(BEDCOVERAGE_LOGS.glob('wes_bedcoverage_*.log')):
    print(path.name)

## 3. ClinCNV run definitions

The two BedCoverage-based ClinCNV runs are defined by the two `run_clincnv.sh` scripts below.

Shared settings:

- `--scoreS 200`
- `--lengthS 9`
- `--filterStep 2`
- `--colNum 4`
- WES on-target and off-target BedCoverage matrices

Difference between the runs:

- `full_wes_s200_l9_f2_baf_bedcoverage_wes` includes `--bafFolder /mnt/myvolume/panel_seq/new_bed_analysis/baf_from_pair_vcfs_clean`
- `full_wes_s200_l9_f2_no_baf_bedcoverage_wes` omits `--bafFolder`

In [ ]:
print('BAF-backed ClinCNV run script:')
show_text(BAF_RUN_SCRIPT)

print('\nNo-BAF ClinCNV run script:')
show_text(NO_BAF_RUN_SCRIPT)

In [ ]:
run_summary = pd.DataFrame([
    summarize_file(BafRun / 'clinCNV.log'),
    summarize_file(BafRun / 'ontargetTumor.summary.xls'),
    summarize_file(BafRun / 'offtargetTumor.summary.xls'),
    summarize_file(BafRun / 'clusterization_of_samples.tsv'),
    summarize_file(NoBafRun / 'clinCNV.log'),
    summarize_file(NoBafRun / 'ontargetTumor.summary.xls'),
    summarize_file(NoBafRun / 'offtargetTumor.summary.xls'),
    summarize_file(NoBafRun / 'clusterization_of_samples.tsv'),
])
run_summary

## 4. Minimal reproducibility command list

These are the local commands that correspond to the scripts documented above.

```bash
# 1. Build the cleaned BAF folder from pair VCFs
bash /mnt/myvolume/panel_seq/new_bed_analysis/baf_from_pair_vcfs.sh

# 2. Rebuild the WES off-target BED (100 kb bins, remove bins < 50 kb)
bash /mnt/myvolume/panel_seq/new_bed_analysis/gen_wes_offtarget_bed_100kb.sh

# 3. Build BedCoverage-derived WES matrices
bash /mnt/myvolume/panel_seq/new_bed_analysis/build_wes_bedcoverage_matrices.sh

# 4a. Run ClinCNV with the cleaned BAF folder
bash /mnt/myvolume/panel_seq/new_bed_analysis/new_clincnv_runs/full_wes_s200_l9_f2_baf_bedcoverage_wes/run_clincnv.sh

# 4b. Run ClinCNV without a BAF folder
bash /mnt/myvolume/panel_seq/new_bed_analysis/new_clincnv_runs/full_wes_s200_l9_f2_no_baf_bedcoverage_wes/run_clincnv.sh
```

If you want to share this notebook externally, these absolute paths make it explicit which local files were used.